In [4]:
import geopandas as gpd
import pandas as pd 

In [18]:
retail_centers = gpd.read_file("geo/postworkshopcentres.gpkg")
retail_centers = retail_centers[['RC_ID', 'geometry']]
retail_centers

,RC_ID,geometry
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420...."
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111...."
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3..."
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983...."
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9..."
...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610...."
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6..."
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5..."
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377...."


# 1. spatial join wiith LAD

In [4]:
lad = gpd.read_file("geo/LA/Local_Authority_Districts_May_2024_Boundaries_UK_BFE_-5758551109064458912/LAD_MAY_2024_UK_BFE.shp")
lad = lad.to_crs(retail_centers.crs)
lad.head()

,LAD24CD,LAD24NM,LAD24NMW,BNG_E,BNG_N,LONG,LAT,geometry
0,E06000001,Hartlepool,None,447161,531473,-1.27017,54.6761,"POLYGON ((447213.9 537036.104, 447228.798 5370..."
1,E06000002,Middlesbrough,None,451141,516887,-1.21099,54.5447,"POLYGON ((448489.897 522071.798, 448592.597 52..."
2,E06000003,Redcar and Cleveland,None,464330,519596,-1.00656,54.5675,"POLYGON ((455553.038 528401.602, 455732.806 52..."
3,E06000004,Stockton-on-Tees,None,444940,518179,-1.30664,54.5569,"POLYGON ((444157.002 527956.304, 444165.898 52..."
4,E06000005,Darlington,None,428029,515648,-1.56835,54.5353,"POLYGON ((423496.602 524724.299, 423497.204 52..."


In [5]:
# LAD join
rc_lad = gpd.sjoin(
    retail_centers,
    lad,
    how='left',
    predicate='within'
)
rc_lad.head()

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LocalAuthorityNM_3,geometry,index_right,LAD24CD,LAD24NM,LAD24NMW,BNG_E,BNG_N,LONG,LAT
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,...,None,"MULTIPOLYGON (((440445.739 367018.091, 440420....",74.0,E07000038,North East Derbyshire,None,437369.0,362955.0,-1.44253,53.1624
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,...,None,"MULTIPOLYGON (((439116.777 363407.888, 439111....",74.0,E07000038,North East Derbyshire,None,437369.0,362955.0,-1.44253,53.1624
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,...,None,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",48.0,E06000052,Cornwall,None,212497.0,64493.0,-4.64254,50.4502
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,...,None,"MULTIPOLYGON (((439008.418 364010.854, 438983....",74.0,E07000038,North East Derbyshire,None,437369.0,362955.0,-1.44253,53.1624
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,...,None,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",70.0,E07000034,Chesterfield,None,440049.0,373358.0,-1.40115,53.2557


In [7]:
# dealing with unmerged
# 1. Identify rows that failed the first join
unmerged_mask = rc_lad['LAD24NM'].isna()
unmerged = rc_lad[unmerged_mask].copy()

# 2. Generate centroids
unmerged_centroids = unmerged.loc[:, : 'geometry'].copy() 
unmerged_centroids['geometry'] = unmerged_centroids.geometry.centroid

# 3. Perform the centroid spatial join
# Now bringing in both Name AND Code
centroid_join = gpd.sjoin(
    unmerged_centroids[['RC_ID', 'geometry']],
    lad[['LAD24NM', 'LAD24CD', 'geometry']], 
    how='left',
    predicate='within'
)

# 4. Update the original dataframe
rc_lad = rc_lad.set_index('RC_ID')
centroid_join = centroid_join.set_index('RC_ID')

# Use 'update' to fill in missing values for both columns
rc_lad['LAD24NM'].update(centroid_join['LAD24NM'])
rc_lad['LAD24CD'].update(centroid_join['LAD24CD'])

# Reset index to bring RC_ID back as a column
rc_lad = rc_lad.reset_index()

# manual fix for RC_ID = 7183, just use the LocalAuthorityNM_1 value
rc_lad.loc[
    rc_lad['RC_ID'] == 7183,
    'LAD24NM'
] = rc_lad.loc[
    rc_lad['RC_ID'] == 7183,
    'LocalAuthorityNM_1'
].values[0]

/var/folders/55/fvwvclfd7t161kfvx0y7f0hm0000gn/T/ipykernel_56470/569619792.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rc_lad['LAD24NM'].update(centroid_join['LAD24NM'])
/var/folders/55/fvwvclfd7t161kfvx0y7f0hm0000gn/T/ipykernel_56470/569619792.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values al

In [8]:
rc_lad['LAD24NM'].isna().sum()

np.int64(0)

In [ ]:
#rc_lad.to_csv("output/rc_lad_lookups.csv", index=False)

In [ ]:
# Checking with exisit LAD names
diff_rows = rc_lad[
    rc_lad['LocalAuthorityNM_1'] != rc_lad['LAD24NM']
]
diff_rows[['RC_ID', 'LocalAuthorityNM_1', 'LocalAuthorityNM_3','LAD24NM']]

# 2. spatial join with the ceremonial counties
- uk bounderies from OS boundery line: https://osdatahub.os.uk/data/downloads/open/BoundaryLine

In [10]:
counties = gpd.read_file("geo/Counties/uk_27700.shp")
counties = counties.to_crs(retail_centers.crs)
counties = counties[['geometry', 'Name']]
counties['Name'] = counties["Name"].str.capitalize()
counties.rename(columns={"Name": "CountyName"}, inplace=True)
counties

,geometry,CountyName
0,"POLYGON ((527179.004 248224.996, 527162.003 24...",Bedfordshire
1,"POLYGON ((504919.202 178392.095, 504918.798 17...",Berkshire
2,"POLYGON ((505704.8 185453.304, 505704.8 185448...",Buckinghamshire
3,"POLYGON ((571833.397 261707.896, 571813.898 26...",Cambridgeshire
4,"POLYGON ((401780.298 370227.103, 401778.6 3702...",Cheshire
...,...,...
92,"MULTIPOLYGON (((144481.685 524629.211, 144480....",Antrim
93,"POLYGON ((118270.039 521388.294, 118401.561 52...",Armagh
94,"POLYGON ((32768.071 538761.678, 32777.298 5387...",Fermanagh
95,"POLYGON ((103000.004 599364.837, 103000.072 59...",Londonderry


In [12]:
# Counties
rc_counties = gpd.sjoin(
    rc_lad[['RC_ID', 'geometry','LAD24NM']],
    counties,
    how='left',
    predicate='within'
)
rc_counties

,RC_ID,geometry,LAD24NM,index_right,CountyName
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",North East Derbyshire,5.0,Derbyshire
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",North East Derbyshire,5.0,Derbyshire
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",Cornwall,89.0,Cornwall
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",North East Derbyshire,5.0,Derbyshire
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",Chesterfield,5.0,Derbyshire
...,...,...,...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610....","Armagh City, Banbridge and Craigavon",93.0,Armagh
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6...",Braintree,37.0,Essex
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5...","Bournemouth, Christchurch and Poole",34.0,Dorset
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377....",Glasgow City,30.0,City of glasgow


In [14]:
C_unmerged = rc_counties[rc_counties['CountyName'].isna()].copy()
C_unmerged_centroids = C_unmerged.copy()
C_unmerged_centroids['geometry'] = C_unmerged_centroids.geometry.centroid
C_centroid_join = gpd.sjoin(
    C_unmerged_centroids[['RC_ID', 'geometry']],
    counties,
    how='left',
    predicate='within'
)
C_centroid_join

,RC_ID,geometry,index_right,CountyName
22,1521,POINT (446199.472 356956.194),5,Derbyshire
96,8540,POINT (440254.947 1139425.485),88,Shetland
184,1693,POINT (212595.038 51707.681),89,Cornwall
433,206,POINT (180816.507 32777.33),89,Cornwall
468,489,POINT (481585.123 389685.535),43,Lincolnshire
...,...,...,...,...
9026,4145,POINT (335993.525 415659.947),44,Merseyside
9130,3667,POINT (347019.397 176562.032),46,Somerset
9270,6009,POINT (450908.153 334225.761),11,Nottinghamshire
9287,4151,POINT (231649.84 674772.669),75,Renfrewshire


In [15]:
# Update the original dataframe
rc_counties = rc_counties.set_index('RC_ID')
C_centroid_join = C_centroid_join.set_index('RC_ID')

# Use 'update' to fill in missing values for both columns
rc_counties['CountyName'].update(C_centroid_join['CountyName'])

# Reset index to bring RC_ID back as a column
rc_counties = rc_counties.reset_index()


/var/folders/55/fvwvclfd7t161kfvx0y7f0hm0000gn/T/ipykernel_56470/1873046375.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rc_counties['CountyName'].update(C_centroid_join['CountyName'])


In [16]:
rc_counties['CountyName'].isna().sum()

np.int64(0)

In [24]:
rc_counties.shape

(9622, 4)

In [ ]:
# rc_counties
rc_counties = rc_counties[['RC_ID', 'geometry', 'LAD24NM', 'CountyName']]
#rc_counties.to_csv("output/rc_county_lookups.csv", index=False)

In [61]:
rc_counties

,RC_ID,geometry,LAD24NM,CountyName
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",North East Derbyshire,Derbyshire
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",North East Derbyshire,Derbyshire
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",Cornwall,Cornwall
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",North East Derbyshire,Derbyshire
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",Chesterfield,Derbyshire
...,...,...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610....","Armagh City, Banbridge and Craigavon",Armagh
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6...",Braintree,Essex
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5...","Bournemouth, Christchurch and Poole",Dorset
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377....",Glasgow City,City of glasgow


# 3. spatial join with the Region

In [66]:
# region
GB_regions= gpd.read_file("geo/Regions/GB_regions.shp")
GB_regions = GB_regions.to_crs(retail_centers.crs)
GB_regions = GB_regions[['geometry', 'Name']]
GB_regions.rename(columns={"Name": "RegionName"}, inplace=True)
GB_regions = GB_regions[GB_regions['RegionName'] != "England"] # drop entire England region

In [67]:
GB_regions['RegionName'].value_counts()

RegionName
Scotland                    2696
Wales                          1
North East                     1
North West                     1
Yorkshire and The Humber       1
East Midlands                  1
West Midlands                  1
East of England                1
London                         1
South East                     1
South West                     1
Name: count, dtype: int64

In [68]:
rc_counties_region = gpd.sjoin(rc_counties, GB_regions, predicate='within', how='left')
rc_counties_region 

,RC_ID,geometry,LAD24NM,CountyName,index_right,RegionName
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",North East Derbyshire,Derbyshire,2701.0,East Midlands
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",North East Derbyshire,Derbyshire,2701.0,East Midlands
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",Cornwall,Cornwall,2706.0,South West
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",North East Derbyshire,Derbyshire,2701.0,East Midlands
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",Chesterfield,Derbyshire,2701.0,East Midlands
...,...,...,...,...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610....","Armagh City, Banbridge and Craigavon",Armagh,NaN,NaN
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6...",Braintree,Essex,2703.0,East of England
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5...","Bournemouth, Christchurch and Poole",Dorset,2706.0,South West
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377....",Glasgow City,City of glasgow,2153.0,Scotland


In [69]:
rc_counties_region['RegionName'].isna().sum()


np.int64(450)

In [70]:
rc_centroids = rc_counties_region.copy()
rc_centroids['geometry'] = rc_centroids.geometry.centroid

rc_counties_region_join = gpd.sjoin(
    rc_centroids[['RC_ID', 'geometry']],
    GB_regions,
    how='left',
    predicate='within'
)

region_lookup = (rc_counties_region_join
    .dropna(subset=['RegionName'])
    .set_index('RC_ID')['RegionName']
)

rc_counties_region['RegionName'] = rc_counties_region['RC_ID'].map(region_lookup)


In [71]:
rc_counties_region['RegionName'].isna().sum()

np.int64(237)

In [74]:
# manual check for NI counties
ni_counties = [
    'TYRONE',
    'ANTRIM',
    'ARMAGH',
    'FERMANAGH',
    'LONDONDERRY',
    'DOWN'
]
mask_ni = rc_counties_region['CountyName'].str.upper().isin(ni_counties)

rc_counties_region.loc[mask_ni, 'RegionName'] = 'NI'

rc_counties_region.loc[mask_ni, ['RC_ID', 'CountyName', 'RegionName']].drop_duplicates()


,RC_ID,CountyName,RegionName
14,2366,Fermanagh,NI
56,5387,Tyrone,NI
156,8545,Tyrone,NI
208,7919,Tyrone,NI
243,7004,Tyrone,NI
...,...,...,...
9572,5567,Tyrone,NI
9589,4390,Antrim,NI
9592,3276,Down,NI
9607,4532,Antrim,NI


In [75]:
rc_counties_region['RegionName'].isna().sum()

np.int64(0)

In [105]:
rc_counties_region = rc_counties_region.drop(columns=['index_right'])
rc_counties_region

,RC_ID,geometry,LAD24NM,CountyName,RegionName
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",North East Derbyshire,Derbyshire,East Midlands
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",North East Derbyshire,Derbyshire,East Midlands
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",Cornwall,Cornwall,South West
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",North East Derbyshire,Derbyshire,East Midlands
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",Chesterfield,Derbyshire,East Midlands
...,...,...,...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610....","Armagh City, Banbridge and Craigavon",Armagh,NI
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6...",Braintree,Essex,East of England
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5...","Bournemouth, Christchurch and Poole",Dorset,South West
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377....",Glasgow City,City of glasgow,Scotland


In [106]:
rc_counties_region.to_csv("output/rc_la_county_region_lookups.csv", index=False)

# RC spatial join with the build up area

In [109]:
buildup = gpd.read_file("geo/buildup/buildup.shp")
buildup= buildup.to_crs(retail_centers.crs)
buildup = buildup[['gsscode', 'geometry','name1_text', 'fid']]
buildup.head()

,gsscode,geometry,name1_text,fid
0,S45000832,"MULTIPOLYGON (((191450 733950, 191450 733925, ...",A'Choingheal,1.0
1,S45000833,"MULTIPOLYGON (((208425 776375, 208400 776375, ...",A'Chorpaich,2.0
2,E63010387,"MULTIPOLYGON (((374450 266700, 374450 266725, ...",Abberley,3.0
3,E63007254,"MULTIPOLYGON (((317275 550175, 317250 550175, ...",Abbeytown,4.0
4,E63009334,"MULTIPOLYGON (((407625 324200, 407625 324225, ...",Abbots Bromley,5.0


In [110]:
# buildup join
rc_buildup = gpd.sjoin(
    retail_centers,
    buildup,
    how='left',
    predicate='within'
)
rc_buildup.head()


,RC_ID,geometry,index_right,gsscode,name1_text,fid
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",3164.0,E63008766,Grassmoor,3165.0
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",1735.0,E63008809,Clay Cross,1736.0
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",6279.0,E63013978,Rock (Cornwall),6280.0
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",NaN,NaN,NaN,NaN
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",1617.0,E63008717,Chesterfield,1618.0


In [92]:
rc_buildup['gsscode'].isna().sum()

np.int64(2668)

In [111]:
B_unmerged = rc_buildup[rc_buildup['gsscode'].isna()].copy()
B_unmerged_centroids = B_unmerged.copy()
B_unmerged_centroids['geometry'] = B_unmerged_centroids.geometry.centroid
B_centroid_join = gpd.sjoin(
    B_unmerged_centroids[['RC_ID', 'geometry']],
    buildup,
    how='left',
    predicate='within'
)

# write centroid-join results back into rc_buildup
centroid_lookup = (
    B_centroid_join
    .dropna(subset=['gsscode'])
    .set_index('RC_ID')['gsscode']
)
rc_buildup.loc[
    rc_buildup['gsscode'].isna(),
    'gsscode'
] = rc_buildup.loc[
    rc_buildup['gsscode'].isna(),
    'RC_ID'
].map(centroid_lookup)

In [112]:
rc_buildup['gsscode'].isna().sum()

np.int64(366)

In [113]:
# Spatial join using intersects
B_still_unmerged = rc_buildup[rc_buildup['gsscode'].isna()].copy()
B_intersect_join = gpd.sjoin(
    B_still_unmerged[['RC_ID', 'geometry']],
    buildup[['gsscode', 'geometry']],
    how='left',
    predicate='intersects'
)


In [116]:
lookup = (
    B_intersect_join
    .dropna(subset=['gsscode'])
    .groupby('RC_ID')['gsscode']
    .first()
)# only keeps the first gsscode for each RC_ID

rc_buildup.loc[
    rc_buildup['gsscode'].isna(),
    'gsscode'
] = rc_buildup.loc[
    rc_buildup['gsscode'].isna(),
    'RC_ID'
].map(lookup) 

In [117]:
rc_buildup['gsscode'].isna().sum()


np.int64(328)

In [118]:
rc_counties_region

,RC_ID,geometry,LAD24NM,CountyName,RegionName
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",North East Derbyshire,Derbyshire,East Midlands
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",North East Derbyshire,Derbyshire,East Midlands
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",Cornwall,Cornwall,South West
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",North East Derbyshire,Derbyshire,East Midlands
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",Chesterfield,Derbyshire,East Midlands
...,...,...,...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610....","Armagh City, Banbridge and Craigavon",Armagh,NI
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6...",Braintree,Essex,East of England
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5...","Bournemouth, Christchurch and Poole",Dorset,South West
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377....",Glasgow City,City of glasgow,Scotland


In [119]:
# merge back to rc_counties_region
rc_final = rc_counties_region.merge(
    rc_buildup[['RC_ID', 'gsscode']],
    on='RC_ID',
    how='left'
)
rc_final

,RC_ID,geometry,LAD24NM,CountyName,RegionName,gsscode
0,6025,"MULTIPOLYGON (((440445.739 367018.091, 440420....",North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,"MULTIPOLYGON (((439116.777 363407.888, 439111....",North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,"MULTIPOLYGON (((194309.881 76200.713, 194331.3...",Cornwall,Cornwall,South West,E63013978
3,6026,"MULTIPOLYGON (((439008.418 364010.854, 438983....",North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,"MULTIPOLYGON (((438294.899 371476.62, 438289.9...",Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...
9617,2844,"MULTIPOLYGON (((111614.723 512113.406, 111610....","Armagh City, Banbridge and Craigavon",Armagh,NI,NaN
9618,4025,"MULTIPOLYGON (((578163.676 233984.52, 578158.6...",Braintree,Essex,East of England,E63011054
9619,2676,"MULTIPOLYGON (((401222.307 90489.387, 401196.5...","Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,"MULTIPOLYGON (((255356.563 666668.682, 255377....",Glasgow City,City of glasgow,Scotland,S45001180


In [120]:
# mannual added the NI as buildup area code for those in NI
rc_final.loc[rc_final['RegionName']=='NI', 'gsscode'] = 'NI'

In [121]:
rc_final['gsscode'].isna().sum() # manual varify these are very small number POI of RCs

np.int64(91)

In [127]:
rc_final.to_csv("output/rc_lookups.csv", index=False)

In [ ]:
# export = rc_final[rc_final['gsscode'].isna()]
# export.to_file("output/check_shapefile.shp")

In [126]:
# retail centers type check
rc = gpd.read_file("geo/postworkshopcentres.gpkg")
rc['centre_type'].value_counts()

centre_type
centre         8450
retail_park    1172
Name: count, dtype: int64

In [2]:
import pandas as pd

path_a = "/Users/carolyin/Documents/Document_Work/GeoDS/RetailCenter/output/rc_lookups.csv"
path_b = "/Users/carolyin/Documents/Document_Work/GeoDS/RetailCenter/output/rc_lookups2.csv"

df_a = pd.read_csv(path_a)
df_b = pd.read_csv(path_b)

print("A shape:", df_a.shape)
print("B shape:", df_b.shape)

# check duplicate keys
print("A duplicate RC_ID:", df_a["RC_ID"].duplicated().sum())
print("B duplicate RC_ID:", df_b["RC_ID"].duplicated().sum())

# align on RC_ID
merged = df_a.merge(
    df_b,
    on="RC_ID",
    how="outer",
    suffixes=("_a", "_b"),
    indicator=True
)

# rows only in one file
only_a = merged[merged["_merge"] == "left_only"]
only_b = merged[merged["_merge"] == "right_only"]
print("Only in A:", len(only_a))
print("Only in B:", len(only_b))

# compare common columns (excluding geometry-like)
cols_a = {c for c in df_a.columns if c != "RC_ID"}
cols_b = {c for c in df_b.columns if c != "RC_ID"}
common_cols = sorted(cols_a & cols_b)

diffs = {}
for col in common_cols:
    ca = f"{col}_a"
    cb = f"{col}_b"
    # treat NaNs as equal
    mask = ~(merged[ca].isna() & merged[cb].isna()) & (merged[ca] != merged[cb])
    diffs[col] = merged.loc[mask, ["RC_ID", ca, cb]]

# show summary
print("Columns compared:", common_cols)
print("Diff counts by column:")
for col, d in diffs.items():
    print(f"{col}: {len(d)}")

# view diffs for a specific column
# diffs["RegionName"].head()

# optional: all diffs combined
all_diffs = pd.concat(diffs.values(), ignore_index=True) if diffs else pd.DataFrame()
all_diffs.head()


A shape: (9622, 6)
B shape: (9622, 6)
A duplicate RC_ID: 0
B duplicate RC_ID: 0
Only in A: 0
Only in B: 0
Columns compared: ['CountyName', 'LAD24NM', 'RegionName', 'geometry', 'gsscode']
Diff counts by column:
CountyName: 0
LAD24NM: 1
RegionName: 0
geometry: 0
gsscode: 0


,RC_ID,CountyName_a,CountyName_b,LAD24NM_a,LAD24NM_b,RegionName_a,RegionName_b,geometry_a,geometry_b,gsscode_a,gsscode_b
0,7183,NaN,NaN,Antrim and Newtownabbey,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
BUA11 = gpd.read_file("geo/buildup/Built_up_Areas_Dec_2011_Boundaries_V2_2022_-9186664123850227917.gpkg")
BUA11.head()

/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'geo/buildup/Built_up_Areas_Dec_2011_Boundaries_V2_2022_-9186664123850227917.gpkg' may only be partially supported
  return ogr_read(


,BUA11CD,BUA11NM,BUA_ID,HAS_SD,SD_COUNT,Area,GlobalID,geometry
0,E34005058,Daventry International Rail Freight Terminal,2397,N,0,0.954996,{4E140658-C7C8-429B-A86C-83FA19C486B6},"MULTIPOLYGON (((457149.998 273050, 457199.996 ..."
1,E34005059,Tunstead Works,1846,N,0,0.372497,{79D8AF67-5B2C-4BC3-92F3-10E11EE5E4C6},"MULTIPOLYGON (((409999.998 374250.001, 409950 ..."
2,E34000019,Hartnoll BUA,463,N,0,0.307500,{0E85F693-8016-4FA5-90A6-8C16F585397D},"MULTIPOLYGON (((298450.002 113200.006, 298450...."
3,E34005061,Saxham Business Park,4174,N,0,0.242497,{C21FFC86-CC86-46D1-8FA9-290400D300FF},"MULTIPOLYGON (((581049.999 265299.998, 581049...."
4,E34005063,"Depot, nr Great Shefford",2056,N,0,0.970007,{AFE8BB64-C4DA-436B-90DE-EAF431B83528},"MULTIPOLYGON (((441699.998 175699.996, 441699...."


# dissolving BUA

In [6]:
# ---------- Utility: dissolve new BUA to match previous version ----------
def dissolve_bua_to_match_previous_version(
    bua_new_path: str,
    bua_old_path: str,
    output_path: str = "geo/buildup/buildup_dissolved_aligned.gpkg",
    old_name_col: str = "BUA11NM",
    crs: str = "EPSG:27700"
):
    """
    Dissolve the latest Built-Up Area (BUA) polygons to mimic the older BUA extent.
    """

    import geopandas as gpd
    from pathlib import Path

    if not Path(bua_new_path).exists() or not Path(bua_old_path).exists():
        raise FileNotFoundError("Check your BUA file paths — one or both do not exist.")

    bua_new = gpd.read_file(bua_new_path)
    bua_old = gpd.read_file(bua_old_path)

    bua_new = bua_new.to_crs(crs)
    bua_old = bua_old.to_crs(crs)

    # Spatial join to assign each new polygon to its overlapping old polygon
    print("Matching new BUAs to old version...")
    joined = gpd.sjoin(bua_new, bua_old[[old_name_col, "geometry"]], how="left", predicate="intersects")

    # Dissolve new BUAs by the old name/ID
    print("Dissolving new BUAs to match old boundaries...")
    bua_dissolved = joined.dissolve(by=old_name_col).reset_index()

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    bua_dissolved.to_file(output_path, driver="GPKG")
    print(f"Dissolved BUA layer saved to: {output_path}")

    return bua_dissolved


In [7]:
# ---------- Buildup ----------
# Paths for old and new BUAs
bua_old_path = "geo/buildup/Built_up_Areas_Dec_2011_Boundaries_V2_2022_-9186664123850227917.gpkg"
bua_new_path = "geo/buildup/buildup.shp"

# Dissolve the latest BUA using the 2011 boundaries as reference
buildup = dissolve_bua_to_match_previous_version(
    bua_new_path=bua_new_path,
    bua_old_path=bua_old_path,
    old_name_col="BUA11NM",  # field from old layer
    output_path="geo/buildup/buildup_2025_dissolved_aligned.gpkg"
)

# Add consistent field names for later use
buildup["gsscode"] = buildup["BUA11NM"]
buildup.rename(columns={"BUA11NM": "name1_text"}, inplace=True)

# Ensure CRS and select required columns
buildup = buildup.to_crs(retail_centers.crs)
buildup = buildup[["gsscode", "geometry", "name1_text"]]


/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'geo/buildup/Built_up_Areas_Dec_2011_Boundaries_V2_2022_-9186664123850227917.gpkg' may only be partially supported
  return ogr_read(


Matching new BUAs to old version...
Dissolving new BUAs to match old boundaries...


FieldError: Error adding field 'fid' to layer